# Stage 00a — PatSeer Download

**What this notebook does:**  
Opens a Chrome browser, logs into PatSeer, and iterates through every record in
your search result.  For each patent it clicks the *Drawings* tab and downloads
all available figures — three file types that PatSeer provides:

| Type | Example filename | Meaning |
|------|-----------------|---------|
| `img` | `US20220267016A1_img003.png` | Individual figure crop — PatSeer already separated these, no splitting needed |
| `D`   | `US20220267016A1_D00005.png` | Full drawing sheet — may contain multiple figures per page |
| `FAT` | `US20220267016A1_FAT001.png` | Composite sheet — may contain multiple figures per page |

Raw PatSeer filenames (e.g. `record_0002_US…-20220825-img003.png`) are cleaned
on save: the `record_NNNN_` prefix and `-YYYYMMDD-` date segment are stripped,
remaining hyphens become underscores.

A `{patent_id}_download_manifest.json` is written into each patent folder
recording which files were saved, the publication date, and the record position.

**Where it fits in the pipeline:**
```
00a.1  (audit — identifies what needs re-downloading and writes CSVs)
 ↓
00a  ←  YOU ARE HERE  (quarantine bad folders → download missing/bad patents)
 ↓
00b  (positional matching → _F / _Fu labels)
 ↓
02   (pad + resize to 518×518)
```

**To resume a partial run:** set `START_FROM` in Cell 1 to the last completed
record index and re-run.  Already-downloaded patents are skipped automatically.

---

| Cell | What it does |
|------|--------------|
| 1  | Load config; set `OUTPUT_DIR`, `START_FROM`, `END_AT` |
| 1b | **Quarantine bad folders** — moves fig_NN / wrong-patent folders to `raw/_quarantine/` so the downloader re-downloads them |
| 2  | Rename `record_XXXX` folders → real patent IDs using Excel lookup |
| 3  | Launch the downloader — opens Chrome, prompts for login, then loops all records |
| 4  | Scan `raw/` and print a per-type download coverage report |

In [17]:
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
src_dir   = repo_root / "src"

_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

print(f"config_loader loaded from: {_cl_path}")

for p in [str(repo_root), str(src_dir)]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(src_dir))

cfg = load_config()

# ─── Batch mode ────────────────────────────────────────────────────────────────
# MISSING_BATCH = True  → download into raw/missing_batch/  (for the 296 missing families)
# MISSING_BATCH = False → download into raw/               (normal full-dataset run)
MISSING_BATCH = True

_raw = Path(cfg["paths"]["raw_images"])
OUTPUT_DIR = _raw / "missing_batch" if MISSING_BATCH else _raw
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Run parameters ────────────────────────────────────────────────────────────
# Paste your PatSeer search URL here (copy from browser address bar after running the PNC search)
SEARCH_BASE_URL = "https://app1.patseer.com/#!/result/ad056431-8522-11e8-944f-22000bd445e0/172"

_start_raw  = input("Start from record number [1]: ").strip()
START_FROM  = int(_start_raw) if _start_raw.isdigit() else 1

# Enter the number of results PatSeer found for your PNC query
# (will be different from 1639 — PatSeer may find fewer if some pub numbers aren't indexed)
_total_raw  = input("Total records in THIS PatSeer search: ").strip()
TOTAL_RECORDS = int(_total_raw) if _total_raw.isdigit() else 1

END_AT = None  # set to int to stop early; None = run to TOTAL_RECORDS
# ──────────────────────────────────────────────────────────────────────────────

print(f"Batch mode     : {'MISSING (→ raw/missing_batch/)' if MISSING_BATCH else 'NORMAL (→ raw/)'}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Start from     : {START_FROM}")
print(f"Total records  : {TOTAL_RECORDS}")
print(f"End at         : {END_AT or TOTAL_RECORDS}")


config_loader loaded from: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/src/config_loader.py
Batch mode     : MISSING (→ raw/missing_batch/)
Output dir     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw/missing_batch
Start from     : 1
Total records  : 70
End at         : 70


## Cell 1b — Quarantine bad folders before downloading

Reads the two CSVs produced by **00a1 Cell 9** plus the **"Needs Redownload"** sheet from `1639_DS_audit.xlsx`, and **moves** bad folders into `raw/_quarantine/` — nothing is deleted.

| Source | What it contains |
|--------|-----------------|
| `Needs Redownload` sheet | All-`fig_NN` fallback folders (400 errors during download) |
| `provenance_fail_detail.csv` | Folders whose images belong to the **wrong patent** |
| `provenance_unknown_not_flagged.csv` | Any fig_NN gaps not yet in the Needs Redownload list |

After moving, the downloader sees those folders as missing and re-downloads them cleanly. The quarantine folder lets you inspect or recover anything if needed.

In [18]:
# ── Cell 1b: Quarantine bad folders before downloading ───────────────────────
# Moves folders that need a clean re-download into raw/_quarantine/.
# Nothing is deleted — quarantine can be inspected or emptied manually later.
# Safe to skip if you only want to download NEW (missing) patents.
import shutil
import pandas as pd
from pathlib import Path

raw_dir     = Path(cfg["paths"]["raw_images"])
data_dir    = Path(cfg["paths"]["data"])
aud_xlsx    = Path(repo_root) / "notebooks" / "outputs" / "1639_DS_audit.xlsx"
quarantine  = raw_dir / "_quarantine"
quarantine.mkdir(exist_ok=True)

# ── Collect folder names to quarantine ───────────────────────────────────────
to_move: dict[str, str] = {}   # folder_name → reason

# Source 1: Needs Redownload sheet (bare fig_NN folders — download interrupted)
if aud_xlsx.exists():
    import openpyxl as _opxl
    _sheets = _opxl.load_workbook(aud_xlsx, read_only=True).sheetnames
    if "Needs Redownload" in _sheets:
        df_nr = pd.read_excel(aud_xlsx, sheet_name="Needs Redownload", dtype=str)
        for name in df_nr["folder_name"].dropna().str.strip():
            if name:
                to_move[name] = "bare fig_NN fallback images (download interrupted)"
        print(f"From 'Needs Redownload' sheet          : {len(df_nr)} folders")
    else:
        print("ℹ  No 'Needs Redownload' sheet in audit Excel (0 folders flagged — this is fine).")
else:
    print(f"⚠  {aud_xlsx.name} not found — run 00a1 Cells 3+4 first.")

# Source 2: Provenance FAIL (images from wrong patent)
fail_csv = data_dir / "provenance_fail_detail.csv"
if fail_csv.exists():
    try:
        df_fail = pd.read_csv(fail_csv, dtype=str)
        if not df_fail.empty:
            for name in df_fail["folder_name"].dropna().str.strip():
                if name:
                    to_move[name] = "provenance FAIL — wrong patent's images"
        print(f"From provenance_fail_detail.csv        : {len(df_fail)} folders")
    except pd.errors.EmptyDataError:
        print("From provenance_fail_detail.csv        : 0 folders (empty — all provenance OK)")
else:
    print("provenance_fail_detail.csv not found — run 00a1 Cell 9 first.")

# Source 3: fig_NN gaps not already in Needs Redownload
gap_csv = data_dir / "provenance_unknown_not_flagged.csv"
if gap_csv.exists():
    try:
        df_gap = pd.read_csv(gap_csv, dtype=str)
        if not df_gap.empty:
            for name in df_gap["folder_name"].dropna().str.strip():
                if name:
                    to_move[name] = "fig_NN UNKNOWN — not previously flagged"
        print(f"From provenance_unknown_not_flagged.csv: {len(df_gap)} folders")
    except pd.errors.EmptyDataError:
        print("From provenance_unknown_not_flagged.csv: 0 folders (empty — no gaps)")
else:
    print("provenance_unknown_not_flagged.csv not found — skipping (none expected if audit is clean).")

print(f"\nTotal unique folders to quarantine : {len(to_move)}")

# ── Dry-run ───────────────────────────────────────────────────────────────────
present      = [(n, r) for n, r in to_move.items() if (raw_dir / n).exists()]
already_gone = [n for n in to_move if not (raw_dir / n).exists()]

print(f"\nDRY-RUN PREVIEW")
print(f"{'─'*60}")
print(f"  Exist on disk (will be moved to _quarantine/) : {len(present)}")
print(f"  Already gone (no action needed)               : {len(already_gone)}")

if present:
    print(f"\n  Sample (first 20):")
    for name, reason in present[:20]:
        imgs = len(list((raw_dir / name).glob("*.png")))
        print(f"    {name:<45}  ({imgs:>3} imgs)  [{reason}]")
    if len(present) > 20:
        print(f"    … and {len(present) - 20} more")

print(f"\n{'═'*60}")
if not present:
    print("Nothing to move — all folders already quarantined or gone. Proceed to Cell 2.")
else:
    confirm = input(
        f"\nType MOVE (all caps) to move {len(present)} folders into _quarantine/, "
        "or anything else to abort: "
    ).strip()

    if confirm == "MOVE":
        moved  = []
        errors = []
        for name, reason in present:
            src = raw_dir / name
            dst = quarantine / name
            if dst.exists():
                # Already in quarantine from a previous run — skip
                print(f"  ~ {name}  (already in _quarantine/, skipping)")
                moved.append(name)
                continue
            try:
                shutil.move(str(src), str(dst))
                print(f"  ✓ Moved  {name}")
                moved.append(name)
            except Exception as e:
                errors.append((name, str(e)))
                print(f"  ✗ ERROR  {name}: {e}")

        print(f"\n{'═'*60}")
        print(f"Moved    : {len(moved)}")
        print(f"Errors   : {len(errors)}")
        if errors:
            for n, e in errors:
                print(f"  {n}: {e}")
        print(f"\nFolders quarantined → {quarantine}")
        print("Proceed to Cell 2 (rename record_ folders) then Cell 3 (download).")
    else:
        print("Aborted — nothing moved.")


ℹ  No 'Needs Redownload' sheet in audit Excel (0 folders flagged — this is fine).
From provenance_fail_detail.csv        : 0 folders
From provenance_unknown_not_flagged.csv: 3 folders

Total unique folders to quarantine : 3

DRY-RUN PREVIEW
────────────────────────────────────────────────────────────
  Exist on disk (will be moved to _quarantine/) : 0
  Already gone (no action needed)               : 3

════════════════════════════════════════════════════════════
Nothing to move — all folders already quarantined or gone. Proceed to Cell 2.


In [19]:
# ── Rename record_XXXX folders → real patent ID using Excel lookup ────────────
# IMPORTANT: This only works correctly if the PatSeer download AND the Excel
# export used the SAME sort order.  Always use 'Document No ↑' for both.
# If sort orders differ, record_position will map to the wrong patent.
#
# For existing record_ folders from an unknown sort run, use PNC re-download
# instead: the audit notebook generates the PNC query for missing patents.
#
# Safe to re-run: already-renamed folders are skipped automatically.

import re, json
import pandas as pd

EXCEL_PATH = Path(cfg["paths"]["patseer_excel"])
raw_dir    = Path(cfg["paths"]["raw_images"])

if not raw_dir.exists():
    print("raw/ folder does not exist yet — nothing to migrate.")
elif not EXCEL_PATH.exists():
    print(f"Excel not found: {EXCEL_PATH}")
else:
    df_excel = pd.read_excel(EXCEL_PATH, usecols=[0])
    print(f"Excel loaded: {len(df_excel)} rows  |  scanning for record_XXXX folders…")
    print("⚠  Only run this if the download and Excel export both used 'Document No ↑' sort.\n")

    migrated, skipped, conflicts = [], [], []

    for folder in sorted(raw_dir.iterdir()):
        if not folder.is_dir() or not re.match(r"record_\d+$", folder.name):
            continue

        manifests = list(folder.glob("*manifest*.json"))
        rec_pos   = None
        if manifests:
            try:
                data    = json.loads(manifests[0].read_text())
                rec_pos = data.get("record_position")
            except Exception:
                pass

        patent_id = None
        if rec_pos is not None:
            row_idx = int(rec_pos) - 1
            if 0 <= row_idx < len(df_excel):
                val = str(df_excel.iloc[row_idx, 0]).strip()
                if val and val.lower() not in ("nan", "none", ""):
                    patent_id = val

        if not patent_id:
            print(f"  ? {folder.name}  (rec_pos={rec_pos}) — not in Excel, skipping")
            skipped.append(folder.name)
            continue

        dest_folder = raw_dir / patent_id
        if dest_folder.exists():
            # Collision: a folder with this patent ID was already downloaded properly.
            # The record_ folder's images are likely duplicates or from wrong sort order.
            existing_imgs = len(list(dest_folder.glob("*.png")))
            record_imgs   = len(list(folder.glob("*.png")))
            print(f"  ⚠ {folder.name} → {patent_id} already exists "
                  f"(record has {record_imgs} imgs, existing has {existing_imgs} imgs) — skipping")
            conflicts.append((folder.name, patent_id))
            continue

        record_prefix = folder.name + "_"
        for f in list(folder.glob("*")):
            if f.name.startswith(record_prefix):
                f.rename(folder / f.name[len(record_prefix):])

        mf_path = folder / "download_manifest.json"
        if mf_path.exists():
            mf_data = json.loads(mf_path.read_text())
            mf_data["patent_id"] = patent_id
            for key in ("img_files", "d_files", "fat_files"):
                mf_data[key] = [
                    n[len(record_prefix):] if n.startswith(record_prefix) else n
                    for n in mf_data.get(key, [])
                ]
            (folder / f"{patent_id}_download_manifest.json").write_text(
                json.dumps(mf_data, indent=2))
            mf_path.unlink(missing_ok=True)

        folder.rename(dest_folder)
        print(f"  ✓ {folder.name}  →  {patent_id}  (Excel row {rec_pos})")
        migrated.append((folder.name, patent_id))

    print(f"\nMigrated : {len(migrated)}")
    print(f"Conflicts: {len(conflicts)}  ← these record_ folders may have wrong patent ID")
    print(f"Unknown  : {len(skipped)}")


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Excel loaded: 1639 rows  |  scanning for record_XXXX folders…
⚠  Only run this if the download and Excel export both used 'Document No ↑' sort.


Migrated : 0
Conflicts: 0  ← these record_ folders may have wrong patent ID
Unknown  : 0


In [20]:
import importlib
import patseer_downloader as dl
dl = importlib.reload(dl)  # pick up source changes without restarting the kernel

# Patch module-level configuration before launching
dl.OUTPUT_DIR     = OUTPUT_DIR
dl.START_FROM     = START_FROM
dl.TOTAL_RECORDS  = END_AT if END_AT is not None else TOTAL_RECORDS
dl.EXCEL_PATH       = Path(cfg["paths"]["patseer_excel"])  # used to resolve record_ folders
dl.SEARCH_BASE_URL  = SEARCH_BASE_URL

# A Chrome window will open — log in to PatSeer and open the record you want
# in Details View.  The download only starts after you press Enter here AND
# the browser is actually on the Details View page (otherwise it re-prompts).
# clean_filename() is applied to every saved file inside download_images().
dl.main()



  Chrome opened. Loading your PatSeer search…

  ⚠  Login page detected — please log in in the Chrome window.
  ✓ Logged in — navigating to search URL…
  ✓ Logged in — navigating to search URL…
  ✓ Logged in — navigating to search URL…
  ✓ Logged in — navigating to search URL…
  ✓ Logged in — navigating to search URL…
  ✓ Search loaded — proceeding.
  Starting from record 1…


[  1/70]
  Patent : IN524958A1
  Manifest exists — skipping

[  2/70]
  Patent : DE102024127443A1
  Manifest exists — skipping

[  3/70]
  Patent : DE102024122189A1
  Manifest exists — skipping

[  4/70]
  Patent : CN120922349A
  Images : 8 found
      ✓ CN120922349APAFP.png  (36 kB)
      ✓ CN120922349A_202511248622.png  (36 kB)
      ✓ CN120922349A_HDA0005578514980000021.png  (41 kB)
      ✓ CN120922349A_HDA0005578514980000022.png  (19 kB)
      ✓ CN120922349A_HDA0005578514980000031.png  (51 kB)
      ✓ CN120922349A_HDA0005578514980000041.png  (29 kB)
      ✓ CN120922349A_HDA0005578514980000042.png  (19 kB)
  

In [21]:
import re

raw_dir = OUTPUT_DIR

patents_attempted  = 0
patents_with_img   = 0
patents_with_d     = 0
patents_with_fat   = 0
total_img          = 0
total_d            = 0
total_fat          = 0

for patent_dir in sorted(raw_dir.iterdir()):
    if not patent_dir.is_dir():
        continue
    patents_attempted += 1

    imgs  = [p for p in patent_dir.glob("*.png") if "_img" in p.name.lower()]
    dfiles = [p for p in patent_dir.glob("*.png") if re.search(r"_d\d{4,}", p.name, re.IGNORECASE)]
    fats  = [p for p in patent_dir.glob("*.png") if "_fat" in p.name.lower()]

    if imgs:   patents_with_img += 1
    if dfiles: patents_with_d   += 1
    if fats:   patents_with_fat += 1

    total_img += len(imgs)
    total_d   += len(dfiles)
    total_fat += len(fats)

def _pct(n):
    return f"{n / patents_attempted * 100:.0f}%" if patents_attempted else "N/A"

avg_imgs = total_img / patents_with_img if patents_with_img else 0

print("══════════════════════════════════════")
print("Download Coverage Report")
print("══════════════════════════════════════")
print(f"Patents attempted      : {patents_attempted}")
print(f"Patents with img files : {patents_with_img}  ({_pct(patents_with_img)})")
print(f"Patents with D files   : {patents_with_d}  ({_pct(patents_with_d)})")
print(f"Patents with FAT files : {patents_with_fat}  ({_pct(patents_with_fat)})")
print(f"Total img files        : {total_img}")
print(f"Total D files          : {total_d}")
print(f"Total FAT files        : {total_fat}")
print(f"Average imgs/patent    : {avg_imgs:.1f}")
print("══════════════════════════════════════")

══════════════════════════════════════
Download Coverage Report
══════════════════════════════════════
Patents attempted      : 179
Patents with img files : 16  (9%)
Patents with D files   : 7  (4%)
Patents with FAT files : 0  (0%)
Total img files        : 86
Total D files          : 59
Total FAT files        : 0
Average imgs/patent    : 5.4
══════════════════════════════════════
